# Solving the OCP via indirect approach, SODE low thrust

first order eqs:
$$
\dot{x} = v_x, \dot{y}
$$


In [ ]:
import sympy
import numpy as np
import function_definitions as field_funcs
import helper_funcs as hfct
import pickle
import scipy
import scipy.optimize as opt
import OCP_construction_functions as OCP_funcs

In [ ]:
####################################################
####################################################
## Modify data here
####################################################
####################################################

good_initial_guess = True
alpha_choice = 1
gamma_choice = 1 
iterator = 0# 0-11

####################################################
####################################################
## end Modify data here
####################################################
####################################################
N_choice= [50,100,150,200,250,300,350,400,450,500,550,600][iterator]

OCP_parameters = {
        # Generic OCP params, always needed
        "T": 28.,   #s
        # "dim_q": 2,  # state dimension will be set by the control Lagrangian
        "dim_u": 1,  # actuation dimension will be set by the control Lagrangian
        "N":N_choice,     #240, 160, 80, 40, 20
        "q0": sympy.Matrix([4.,0.]),        #(m,1)
        "dq0": sympy.Matrix([0.,4.]),       #(m/s,1/s)
        "qT": sympy.Matrix([5.,0.]),       #(r,phi) #mayer term does not consider phiT, rather automatically uses the one of the cartesian case
        "dqT": sympy.Matrix([0.,0]),       #(vr,vphi)
        "alpha": alpha_choice,
        "gamma": gamma_choice,
        'folder_name': 'low_thrust/polar_variables',
        'RK_b': sympy.Matrix([1]),
        'RK_c': sympy.Matrix([1/2]),
        'RK_a': sympy.Matrix([[1/2]]),
        #specific to model
        "m":1.,  # kg
        "M":10,
        "G": 1., #m/s**2
        "Aq": 1.0, #  kg/s^2
        "Adq": 1.0, # kg/s
        "variable_names": ["r", "phi"]
    }
OCP_parameters["dqT"][1] =sympy.sqrt(OCP_parameters["G"]*OCP_parameters["M"]/OCP_parameters["qT"][0]**3)
OCP_parameters["dq0"][1] =sympy.sqrt(OCP_parameters["G"]*OCP_parameters["M"]/OCP_parameters["q0"][0]**3)
OCP_parameters["h"] =  OCP_parameters["T"]*1.0/N_choice 
OCP_parameters["times"] = sympy.Matrix([i*OCP_parameters["h"] for i in range(OCP_parameters["N"]+1)])
if not good_initial_guess:
    OCP_parameters["folder_name"] += '_bad_initial_guess'


# Function definition

In [ ]:
continuous_eq = OCP_funcs.Direct_continuous_generator(OCP_parameters,f_func=field_funcs.f_vec_polar,rho_func=field_funcs.rho_vec_polar,running_cost_func=field_funcs.running_cost_polar,mayer_func=field_funcs.mayer_term_polar,g_func=field_funcs.g_mat_polar)

# initial guess creation

In [ ]:
OCP_parameters_np_version = hfct.sympy_to_np_dict(OCP_parameters,alpha_choice,gamma_choice)

if good_initial_guess:
    save_data_file = 'data/low_thrust/polar_variables/reference/polar_ref_newer.pkl'

    alpha_value,beta_value,gamma_value=alpha_choice,gamma_choice,gamma_choice
    def dphi_from_radius(radius,grav,Mass):
        return  np.sqrt(grav* Mass/radius**3)


    with open(save_data_file, 'rb') as files:
        initial_guess_data = pickle.load(files)

    q_d_start_guess = np.array(initial_guess_data['q_d_new'])
    lam_d_start_guess =np.array(initial_guess_data['lambda_d_new'])
    lamq_d_start_guess =np.array(initial_guess_data['lambda_q_standard'])
    lamv_d_start_guess =np.array(initial_guess_data['lambda_v_standard'])
    v_standard_start_guess= np.array(initial_guess_data["vq_d_standard"])
    u_d_start_guess = initial_guess_data['u_d_new']

    cs_q =  scipy.interpolate.CubicSpline(np.linspace(0,OCP_parameters_np_version["T"],len(q_d_start_guess)), q_d_start_guess.reshape([len(q_d_start_guess),2]))
    cs_lam = scipy.interpolate.CubicSpline(np.linspace(0,OCP_parameters_np_version["T"],len(lam_d_start_guess)), lam_d_start_guess.reshape([len(lam_d_start_guess),2]))  
    cs_u= scipy.interpolate.CubicSpline(np.linspace(0,OCP_parameters_np_version["T"],len(u_d_start_guess)),u_d_start_guess)
    cs_vq= scipy.interpolate.CubicSpline(np.linspace(0,OCP_parameters_np_version["T"],len(v_standard_start_guess)),v_standard_start_guess)
    cs_lamq= scipy.interpolate.CubicSpline(np.linspace(0,OCP_parameters_np_version["T"],len(lamq_d_start_guess)),lamq_d_start_guess)
    cs_lamv= scipy.interpolate.CubicSpline(np.linspace(0,OCP_parameters_np_version["T"],len(lamv_d_start_guess)),lamv_d_start_guess)

    U_d_1_use = cs_u(np.array(OCP_parameters_np_version["times"]) + OCP_parameters_np_version["gamma"]*OCP_parameters_np_version["h"])
    U_d_2_use = cs_u(np.array(OCP_parameters_np_version["times"]) + (1-OCP_parameters_np_version["gamma"])*OCP_parameters_np_version["h"])

    q_d_use = cs_q(OCP_parameters_np_version["times"])
    lambda_d_use = cs_lam(OCP_parameters_np_version["times"])
    lambda_q_d_use = cs_lamq(OCP_parameters_np_version["times"])
    lambda_v_d_use = cs_lamv(OCP_parameters_np_version["times"])
    v_q_d_use = cs_vq(OCP_parameters_np_version["times"])

    mu_use = np.array([0.1,0.1])
    nu_use = np.array([0.1,0.1])



In [ ]:
def polar_to_cartesian_sympy(q_polar):
    r,phi = q_polar
    q_cartesian = sympy.Matrix([r * sympy.cos(phi), r * sympy.sin(phi)])
    return q_cartesian

def polar_to_cartesian_velocity(q_polar,v_polar):
    jacobian = polar_to_cartesian_sympy(q_polar).jacobian(q_polar)
    v_cartesian = jacobian@ v_polar
    return v_cartesian
#alterationsHere
def polar_to_cartesian_lambda_q(q_polar,v_polar,lamq_polar,lamv_polar):
    # testmat = sympy.Matrix([[q_polar[0],0],[0,1+0*q_polar[0]]])
    # jacobianT = ((polar_to_cartesian_sympy(q_polar).jacobian(q_polar))@testmat).T
    jacobianT = ((polar_to_cartesian_sympy(q_polar).jacobian(q_polar))).T
    jacobianinv = sympy.inv_quick(jacobianT)
    r,phi=q_polar
    vr,vphi=v_polar
    lamr,lamphi=lamq_polar
    lamvr,lamvphi=lamv_polar
    additional_term_x = - vphi *sympy.sin(phi) *lamvr -(vphi*sympy.cos(phi)/r- vr*sympy.sin(phi)/r**2)*lamvphi
    additional_term_y = vphi*sympy.cos(phi)*lamvr - (vphi*sympy.sin(phi)/r+ vr*sympy.cos(phi)/r**2)*lamvphi
    additional_term = sympy.Matrix([additional_term_x,additional_term_y])
    return jacobianinv @ lamq_polar + additional_term


def polar_to_cartesian_lambda_v(q_polar,v_polar,lamq_polar,lamv_polar):
    base_jacobian = polar_to_cartesian_sympy(q_polar).jacobian(q_polar)
    G_v = base_jacobian@ v_polar
    jacobianT = G_v.jacobian(v_polar).T
    jacobianinv = sympy.inv_quick(jacobianT)

    return jacobianinv @ lamv_polar

# technically should use r,phi, but for ease just use cartesian variables and reinterpret
polar_to_cartesian_trafo_q = polar_to_cartesian_sympy(continuous_eq.q)
polar_to_cartesian_trafo_vq = polar_to_cartesian_velocity(continuous_eq.q,continuous_eq.vq)
polar_to_cartesian_trafo_lamq = polar_to_cartesian_lambda_q(continuous_eq.q,continuous_eq.vq,continuous_eq.lamq,continuous_eq.lamv)
polar_to_cartesian_trafo_lamv = polar_to_cartesian_lambda_v(continuous_eq.q,continuous_eq.vq,continuous_eq.lamq,continuous_eq.lamv)

lambdified_q_p_to_c = sympy.lambdify(continuous_eq.q,polar_to_cartesian_trafo_q)
lambdified_vq_p_to_c = sympy.lambdify(sympy.flatten(continuous_eq.q)+sympy.flatten(continuous_eq.vq),polar_to_cartesian_trafo_vq)
lambdified_lamq_p_to_c = sympy.lambdify(sympy.flatten(continuous_eq.q)+sympy.flatten(continuous_eq.vq)+sympy.flatten(continuous_eq.lamq)+sympy.flatten(continuous_eq.lamv),polar_to_cartesian_trafo_lamq)
lambdified_lamv_p_to_c = sympy.lambdify(sympy.flatten(continuous_eq.q)+sympy.flatten(continuous_eq.vq)+sympy.flatten(continuous_eq.lamq)+sympy.flatten(continuous_eq.lamv),polar_to_cartesian_trafo_lamv)


def polar_to_cartesian(q_polar):
    r,phi = q_polar
    q_cartesian = np.array([r * np.cos(phi), r * np.sin(phi)])
    return q_cartesian

def cartesian_to_polar(q_cartesian):
    x,y = q_cartesian
    r = np.sqrt(x**2 + y**2)
    q_polar = np.array([r, 0])
    if x>0 and y>=0:
        q_polar[1] = np.arccos(x/r)
    elif x<0 and y>=0:
        q_polar[1] =  np.arccos(x/r)
    elif  x<0 and y < 0:
        q_polar[1] = 2*np.pi- np.arccos(x/r)
    else:
        q_polar[1] = 2*np.pi  - np.arccos(x/r)
    return q_polar 


In [ ]:
#initial guess in polar coordinates trafo

if good_initial_guess:
    q_cart_d = q_d_use
    q_polar_d = []
    twopi_ticker = 0

    for i in range(len(q_cart_d)):
        if i >0 and q_cart_d[i][:][0] >=0 and q_cart_d[i][:][1] >= 0 and q_cart_d[i-1][:][1] < 0 and q_cart_d[i-1][:][0] >= 0:
            twopi_ticker += 1
        q_polar_d.append(cartesian_to_polar(q_cart_d[i][:]))
        q_polar_d[-1][1] += 2*(twopi_ticker-1)*np.pi
    q_polar_d = np.array(q_polar_d)
    q_cart_test_d = []
    for el in q_polar_d:
        q_cart_test_d.append(polar_to_cartesian(el))
    q_cart_test_d = np.array(q_cart_test_d)


# Evolution via state eq

In [ ]:
discrete_equations = OCP_funcs.discrete_standard_direct_eq_generator(continuous_eq)

# lambdify the equations for root finding

In [ ]:
#KKT direct approach
standard_direct_midpoint_KKT = discrete_equations.calc_KKT()
lambdified_KKT =  sympy.lambdify(standard_direct_midpoint_KKT[1],standard_direct_midpoint_KKT[0].subs(discrete_equations.h,OCP_parameters_np_version["h"]).subs(discrete_equations.cont_equations.parameters["alpha"],alpha_choice).subs(discrete_equations.cont_equations.parameters["gamma"],gamma_choice)) 
lambdified_KKT_eval = lambda x :lambdified_KKT(*x)


In [ ]:
#KKT direct approach with new Lagrangian
standard_direct_KKT_new = discrete_equations.calc_KKT_new()

lambdified_KKT_new =  sympy.lambdify(standard_direct_KKT_new[1],standard_direct_KKT_new[0].subs(discrete_equations.h,OCP_parameters_np_version["h"]).subs(discrete_equations.cont_equations.parameters["alpha"],alpha_choice).subs(discrete_equations.cont_equations.parameters["gamma"],gamma_choice))

lambdified_KKT_new_eval = lambda x :lambdified_KKT_new(*x)


In [ ]:
# KKT new approach without u. SLow because not efficiently coded
standard_direct_KKT_new_no_u = discrete_equations.calc_KKT_new_no_u()
lambdified_KKT_new_no_u =  sympy.lambdify(standard_direct_KKT_new_no_u[1],standard_direct_KKT_new_no_u[0].subs(discrete_equations.h,OCP_parameters_np_version["h"]).subs(discrete_equations.cont_equations.parameters["alpha"],alpha_choice).subs(discrete_equations.cont_equations.parameters["gamma"],gamma_choice))
lambdified_KKT_new_eval_no_u = lambda x :lambdified_KKT_new_no_u(*x)


In [ ]:
v_y_vec = sympy.Matrix(sympy.symbols("v_r,v_phi,v_lambda_r,v_lambda_phi"))
p_y_vec = sympy.Matrix(sympy.symbols("p_r,p_phi,p_lambda_r,p_lambda_phi"))

p_y_legendre=discrete_equations.cont_equations.p_y()
v_sol = sympy.solve(sympy.Eq(p_y_legendre,p_y_vec),v_y_vec)
v_vec = sympy.Matrix([v_sol[tmp] for tmp in v_y_vec])

v_eval = lambda q,lam,p_y: v_vec.evalf(subs={"r":q[0],"phi":q[1],"lambda_r":lam[0],"lambda_phi":lam[1], "p_r":p_y[0],"p_phi":p_y[1],"p_lambda_r":p_y[2],"p_lambda_phi":p_y[3]})


## Creation of the initial guess

In [ ]:
import copy
if good_initial_guess:
    initial_guess = [0.1,0.1,0.1,0.1] #mu and nu
    initial_guess_new = [0.1,0.1,0.1,0.1]
    initial_guess_new_no_u=[]

    p_y_d_use = np.array(sympy.Matrix([sympy.flatten(tmp) for tmp in discrete_equations.p_v_d_from_y_d(q_d_use[:,:],lambda_d_use[:,:],U_d_1_use[:,:],U_d_2_use[:,:],OCP_parameters_np_version)]),dtype=float)
    v_y_d_use = []
    for tmp1,tmp2,tmp3 in zip(q_d_use[:,:],lambda_d_use[:,:],p_y_d_use):
        v_y_d_use.append(sympy.flatten(v_eval(tmp1,tmp2,tmp3)))
    v_y_d_use = np.array(v_y_d_use) 

    #polar guesses
    for tmp in q_d_use:
        initial_guess += list(tmp)
        initial_guess_new+=list(tmp)
        #vq guess

    # for tmp1, tmp2 in zip(q_polar_d[:-1], q_polar_d[1:]):
        # initial_guess += list((tmp2-tmp1)/OCP_parameters_np_version["h"])  
    initial_guess += list(v_q_d_use.flatten() )
    # initial_guess += [0,0]
    #lambda terms

    for tmp in lambda_d_use:
        initial_guess_new += list(tmp)
    for tmp in lambda_q_d_use:    
        initial_guess += list(tmp)
    for tmp in lambda_v_d_use:    
        initial_guess += list(tmp)

    initial_guess_new_no_u =copy.deepcopy( initial_guess_new)
    initial_guess += list(U_d_1_use.flatten()) 
    initial_guess += list(U_d_2_use.flatten()) 
    initial_guess_new += list(U_d_1_use.flatten()) 
    initial_guess_new += list(U_d_2_use.flatten()) 

    # needed because v_y creates sympy objects...
    initial_guess=np.array(initial_guess,dtype='float')
    initial_guess_new=np.array(initial_guess_new,dtype='float')
    initial_guess_new_no_u=np.array(initial_guess_new_no_u,dtype='float')

# Generation of the 'bad' initial guess

In [ ]:

if not good_initial_guess:
    save_data_file = 'data/start_guess_traj.pkl'

    alpha_value,beta_value,gamma_value=alpha_choice,gamma_choice,gamma_choice

    OCP_parameters_np_version = hfct.sympy_to_np_dict(OCP_parameters,alpha_choice,gamma_choice)

    with open(save_data_file, 'rb') as files:
        initial_guess_data = pickle.load(files)

    q_d_start_guess = np.array(initial_guess_data['q_d'])
    lam_d_start_guess =np.array(initial_guess_data['lam_d'])
    u_d_start_guess = initial_guess_data['U_d']

    cs_q =  scipy.interpolate.CubicSpline(np.linspace(0,OCP_parameters_np_version["T"],len(q_d_start_guess)), q_d_start_guess.reshape([len(q_d_start_guess),2]))
    cs_lam = scipy.interpolate.CubicSpline(np.linspace(0,OCP_parameters_np_version["T"],len(lam_d_start_guess)), lam_d_start_guess.reshape([len(lam_d_start_guess),2]))  
    cs_u= scipy.interpolate.CubicSpline(np.linspace(0,OCP_parameters_np_version["T"],len(u_d_start_guess)),u_d_start_guess)

    U_d_1_use = cs_u(np.array(OCP_parameters_np_version["times"]) + OCP_parameters_np_version["gamma"]*OCP_parameters_np_version["h"]).reshape(len(OCP_parameters_np_version["times"]),1,1)
    U_d_2_use = cs_u(np.array(OCP_parameters_np_version["times"]) + (1-OCP_parameters_np_version["gamma"])*OCP_parameters_np_version["h"]).reshape(len(OCP_parameters_np_version["times"]),1,1)

    q_d_use = cs_q(OCP_parameters_np_version["times"]).reshape(len(OCP_parameters_np_version["times"]),2,1)
    lambda_d_use = cs_lam(OCP_parameters_np_version["times"]).reshape(len(OCP_parameters_np_version["times"]),2,1)
    mu_use = initial_guess_data['mu']
    nu_use = initial_guess_data['nu']
    v_q_use =[]


In [ ]:
import copy
if not good_initial_guess:

    q_cart_d = q_d_use
    q_polar_d = []
    twopi_ticker = 0

    for i in range(len(q_cart_d)):
        if i >0 and q_cart_d[i][:,0][0] >=0 and q_cart_d[i][:,0][1] >= 0 and q_cart_d[i-1][:,0][1] < 0 and q_cart_d[i-1][:,0][0] >= 0:
            twopi_ticker += 1
        q_polar_d.append(cartesian_to_polar(q_cart_d[i][:,0]))
        q_polar_d[-1][1] += 2*(twopi_ticker-1)*np.pi
    q_polar_d = np.array(q_polar_d)
    q_cart_test_d = []
    for el in q_polar_d:
        q_cart_test_d.append(polar_to_cartesian(el))
    q_cart_test_d = np.array(q_cart_test_d)


    initial_guess = [1,1,1,1] #mu and nu
    initial_guess_new = [1,1,1,1]
    initial_guess_new_no_u=[]

    p_y_d_use = np.array(sympy.Matrix([sympy.flatten(tmp) for tmp in discrete_equations.p_v_d_from_y_d(q_d_use[:,:,0],lambda_d_use[:,:,0],U_d_1_use[:,:,0],U_d_2_use[:,:,0],OCP_parameters_np_version)]),dtype=float)
    v_y_d_use = []
    for tmp1,tmp2,tmp3 in zip(q_d_use[:,:,0],lambda_d_use[:,:,0],p_y_d_use):
        v_y_d_use.append(sympy.flatten(v_eval(tmp1,tmp2,tmp3)))
    v_y_d_use = np.array(v_y_d_use) 

    #polar guesses
    #g guess
    for tmp in q_polar_d:
        initial_guess += list(tmp)
        initial_guess_new+=list(tmp)
        #vq guess
    for tmp1, tmp2 in zip(q_polar_d[:-1], q_polar_d[1:]):
        initial_guess += list((tmp2-tmp1)/OCP_parameters_np_version["h"])  
    initial_guess += [0,0]
    #lambda terms
    for tmp in q_polar_d:
        initial_guess += list(tmp)
        initial_guess_new += list(tmp)
    for tmp in q_polar_d:
        initial_guess += list(tmp)
    initial_guess_new_no_u =copy.deepcopy( initial_guess_new)
    initial_guess += list(U_d_1_use.flatten()) 
    initial_guess += list(U_d_2_use.flatten()) 
    initial_guess_new += list(U_d_1_use.flatten()) 
    initial_guess_new += list(U_d_2_use.flatten()) 

    # needed because v_y creates sympy objects...
    initial_guess=np.array(initial_guess,dtype='float')
    initial_guess_new=np.array(initial_guess_new,dtype='float')
    initial_guess_new_no_u=np.array(initial_guess_new_no_u,dtype='float')



In [ ]:
# comparison H, u_d, I calc

u_vec_comparison = []
for i in range(len(q_d_use)):
    u_vec_comparison.append(discrete_equations.cont_equations.u_eval_from_new(q_d_use[i],lambda_d_use[i],p_y_d_use[i][:2],p_y_d_use[i][2:],OCP_parameters))

u_vec_comparison = np.array(u_vec_comparison)

H_control_comparison = []
for i in range(len(q_d_use)):
    H_control_comparison.append(discrete_equations.cont_equations.new_control_H_eval(q_d_use[i],lambda_d_use[i],p_y_d_use[i][:2],p_y_d_use[i][2:],u_vec_comparison[i],OCP_parameters_np_version))
H_control_comparison = np.array(H_control_comparison)  

# root finding with timing for the different schemes

In [ ]:
#Standard scheme
import time
starttime = time.time()
standard_result_polar_KKT = opt.root(lambdified_KKT_eval,x0=initial_guess,method="lm")
endtime = time.time()
Dt_standard = endtime-starttime
dt_standard = Dt_standard/standard_result_polar_KKT.nfev

In [ ]:
#new lagrangian approach
starttime = time.time()

standard_result_polar_KKT_new = opt.root(lambdified_KKT_new_eval,x0=initial_guess_new,method="lm")
endtime = time.time()

Dt_new = endtime-starttime
dt_new = Dt_new/standard_result_polar_KKT_new.nfev

In [ ]:
# #new lagrangian approach, no u
starttime = time.time()

standard_result_polar_KKT_new_no_u = opt.root(lambdified_KKT_new_eval_no_u,x0=initial_guess_new_no_u,method="lm")
# standard_result_polar_KKT_new_no_u = opt.root(lambdified_KKT_new_eval_no_u,x0=standard_result_polar_KKT_new.x[:-(N_choice+1)*2],method="lm")

endtime = time.time()

Dt_new_no_u = endtime-starttime
dt_new_no_u = Dt_new_no_u/standard_result_polar_KKT_new_no_u.nfev

## Some output for fast checking

In [ ]:
print(dt_standard)
print(dt_new)
print(dt_new_no_u)

In [ ]:
standard_result_polar_KKT

In [ ]:
standard_result_polar_KKT_new

In [ ]:
standard_result_polar_KKT_new_no_u

# Formatting the output

In [ ]:
reshape_params = discrete_equations.cont_equations.parameters
reshape_N = reshape_params["N"]
reshape_dim_q = reshape_params["dim_q"]
reshape_dim_u = reshape_params["dim_u"]
mu_KKT,nu_KKT = standard_result_polar_KKT.x[:2*reshape_dim_q].reshape(2,reshape_dim_q)
q_d_KKT = standard_result_polar_KKT.x[2*reshape_dim_q:2*reshape_dim_q + reshape_dim_q * (reshape_N+1)].reshape(reshape_N + 1,reshape_dim_q)
vq_d_KKT = standard_result_polar_KKT.x[2*reshape_dim_q + reshape_dim_q * (reshape_N+1):2*reshape_dim_q + 2*reshape_dim_q * (reshape_N+1)].reshape(reshape_N + 1,reshape_dim_q)
lamq_d_KKT = standard_result_polar_KKT.x[2*reshape_dim_q + 2*reshape_dim_q * (reshape_N+1):2*reshape_dim_q + 3*reshape_dim_q * (reshape_N+1)].reshape(reshape_N + 1,reshape_dim_q)
lamv_d_KKT = standard_result_polar_KKT.x[2*reshape_dim_q + 3*reshape_dim_q * (reshape_N+1):2*reshape_dim_q + 4*reshape_dim_q * (reshape_N+1)].reshape(reshape_N + 1,reshape_dim_q)
U1_d_KKT = standard_result_polar_KKT.x[2*reshape_dim_q + 4*reshape_dim_q * (reshape_N+1):2*reshape_dim_q + 4*reshape_dim_q * (reshape_N+1) + (reshape_N+1)*reshape_dim_u ].reshape(reshape_N + 1,reshape_params["dim_u"])
U2_d_KKT = standard_result_polar_KKT.x[2*reshape_dim_q + 4*reshape_dim_q * (reshape_N+1)+ (reshape_N+1)*reshape_dim_u:].reshape(reshape_N + 1,reshape_params["dim_u"])

u_vec_KKT = []
for i in range(len(q_d_KKT)):
    u_vec_KKT.append(discrete_equations.cont_equations.u_Pontryagin_eval(q_d_KKT[i],vq_d_KKT[i],lamq_d_KKT[i],lamv_d_KKT[i],OCP_parameters_np_version))
u_vec_KKT= np.array(u_vec_KKT)
H_Pontry_KKT = []
for i in range(len(q_d_KKT)):
    H_Pontry_KKT.append(discrete_equations.cont_equations.Pontryagin_H_eval(q_d_KKT[i],vq_d_KKT[i],lamq_d_KKT[i],lamv_d_KKT[i],u_vec_KKT[i],OCP_parameters_np_version))
H_Pontry_KKT = np.array(H_Pontry_KKT)    

In [ ]:
def vk_no_u(qk,lamk,qk1,lamk1,use_p= True):
    y_k = sympy.flatten(discrete_equations.control_L_k_no_u['vars_k'][:-2])
    y_k1 = sympy.flatten(discrete_equations.control_L_k_no_u['vars_k1'][:-2])
    if use_p:
        vqk = discrete_equations.vqk1_p_no_u()
    else:
        vqk = discrete_equations.vqk_m_no_u()
    vqk=vqk.subs([[tmp1,tmp2] for tmp1,tmp2 in zip(y_k,sympy.flatten((qk,lamk)))])
    vqk=vqk.subs([[tmp1,tmp2] for tmp1,tmp2 in zip(y_k1,sympy.flatten((qk1,lamk1)))])
    return vqk



In [ ]:
reshape_params = discrete_equations.cont_equations.parameters
reshape_N = reshape_params["N"]
reshape_dim_q = reshape_params["dim_q"]
reshape_dim_u = reshape_params["dim_u"]
mu_KKT_new,nu_KKT_new = standard_result_polar_KKT_new.x[:2*reshape_dim_q].reshape(2,reshape_dim_q)
q_d_KKT_new = standard_result_polar_KKT_new.x[2*reshape_dim_q:2*reshape_dim_q + reshape_dim_q * (reshape_N+1)].reshape(reshape_N + 1,reshape_dim_q)
lam_d_KKT_new = standard_result_polar_KKT_new.x[2*reshape_dim_q + reshape_dim_q * (reshape_N+1):2*reshape_dim_q + 2*reshape_dim_q * (reshape_N+1)].reshape(reshape_N + 1,reshape_dim_q)

U1_d_KKT_new = standard_result_polar_KKT_new.x[2*reshape_dim_q + 2*reshape_dim_q * (reshape_N+1):2*reshape_dim_q + 2*reshape_dim_q * (reshape_N+1) + (reshape_N+1)*reshape_dim_u].reshape(reshape_N + 1,reshape_params["dim_u"])
U2_d_KKT_new = standard_result_polar_KKT_new.x[2*reshape_dim_q + 2*reshape_dim_q * (reshape_N+1) + (reshape_N+1)*reshape_dim_u:].reshape(reshape_N + 1,reshape_params["dim_u"])

p_y_d_new = np.array([sympy.flatten(tmp) for tmp in discrete_equations.p_v_d_from_y_d(q_d_KKT_new,lam_d_KKT_new,U1_d_KKT_new,U2_d_KKT_new,OCP_parameters_np_version)])



u_vec_KKT_new = []
for i in range(len(q_d_KKT_new)):
    u_vec_KKT_new.append(discrete_equations.cont_equations.u_eval_from_new(q_d_KKT_new[i],lam_d_KKT_new[i],p_y_d_new[i][:2],p_y_d_new[i][2:],OCP_parameters))

u_vec_KKT_new = np.array(u_vec_KKT_new)

H_control_KKT_new = []
for i in range(len(q_d_KKT_new)):
    H_control_KKT_new.append(discrete_equations.cont_equations.new_control_H_eval(q_d_KKT_new[i],lam_d_KKT_new[i],p_y_d_new[i][:2],p_y_d_new[i][2:],u_vec_KKT_new[i],OCP_parameters_np_version))
H_control_KKT_new = np.array(H_control_KKT_new)  

mu_KKT_new_no_u,nu_KKT_new_no_u = standard_result_polar_KKT_new_no_u.x[:2*reshape_dim_q].reshape(2,reshape_dim_q)
q_d_KKT_new_no_u = standard_result_polar_KKT_new_no_u.x[2*reshape_dim_q:2*reshape_dim_q + reshape_dim_q * (reshape_N+1)].reshape(reshape_N + 1,reshape_dim_q)
lam_d_KKT_new_no_u = standard_result_polar_KKT_new_no_u.x[2*reshape_dim_q + reshape_dim_q * (reshape_N+1):2*reshape_dim_q + 2*reshape_dim_q * (reshape_N+1)].reshape(reshape_N + 1,reshape_dim_q)
u_calc_no_u = sympy.lambdify(sympy.flatten(continuous_eq.q)+sympy.flatten(continuous_eq.lamq),continuous_eq.u_lambda_from_newH()[0])

u_d_new_no_u = []
for tmpq,tmplam in zip(q_d_KKT_new_no_u,lam_d_KKT_new_no_u):
    u_d_new_no_u.append(u_calc_no_u(*tmpq,*tmplam))

p_y_d_new_no_u = np.array([sympy.flatten(tmp) for tmp in discrete_equations.p_v_d_from_y_d_no_u(q_d_KKT_new_no_u,lam_d_KKT_new_no_u,OCP_parameters_np_version)])

    

In [ ]:
v_sol = sympy.solve(sympy.Eq(p_y_legendre,p_y_vec),v_y_vec)

v_y_vec = sympy.Matrix(sympy.symbols("v_r,v_phi,v_lambda_r,v_lambda_phi"))
p_y_vec = sympy.Matrix(sympy.symbols("p_r,p_phi,p_lambda_r,p_lambda_phi"))

p_y_legendre=discrete_equations.cont_equations.p_y()
v_sol = sympy.solve(sympy.Eq(p_y_legendre,p_y_vec),v_y_vec)
v_vec = sympy.Matrix([v_sol[tmp] for tmp in v_y_vec])

v_eval = lambda q,lam,p_y: v_vec.evalf(subs={"r":q[0],"phi":q[1],"lambda_r":lam[0],"lambda_phi":lam[1], "p_r":p_y[0],"p_phi":p_y[1],"p_lambda_r":p_y[2],"p_lambda_phi":p_y[3]})


v_y_d_new = []
for tmp1,tmp2,tmp3 in zip(q_d_KKT_new,lam_d_KKT_new,p_y_d_new):
    v_y_d_new.append(sympy.flatten(v_eval(tmp1,tmp2,tmp3)))
v_y_d_new = np.array(v_y_d_new) 
v_y_d_new_no_u = []
for tmp1,tmp2,tmp3 in zip(q_d_KKT_new_no_u,lam_d_KKT_new_no_u,p_y_d_new_no_u):
    v_y_d_new_no_u.append(sympy.flatten(v_eval(tmp1,tmp2,tmp3)))
v_y_d_new_no_u = np.array(v_y_d_new_no_u) 


v_y_d_new_cart = []
lam_d_KKT_new_cart = []
for tmpq,tmplam,tmpvq,tmpvlam in zip(q_d_KKT_new,lam_d_KKT_new,v_y_d_new[:,:2],v_y_d_new[:,2:]):
    cart_vq = np.array(lambdified_vq_p_to_c(*tmpq,*tmpvq)).flatten()
    cart_vlam = np.array(lambdified_lamq_p_to_c(*tmpq,*tmpvq,*tmpvlam,*tmplam)).flatten()
    lam_d_KKT_new_cart.append(np.array(lambdified_lamv_p_to_c(*tmpq,*tmpvq,*tmpvlam,*tmplam)).flatten())
    v_y_d_new_cart.append(list(cart_vq) + list(cart_vlam))

v_y_d_new_cart = np.array(v_y_d_new_cart)
lam_d_KKT_new_cart = np.array(lam_d_KKT_new_cart)

v_y_d_new_no_u_cart = []
lam_d_KKT_new_no_u_cart = []
for tmpq,tmplam,tmpvq,tmpvlam in zip(q_d_KKT_new_no_u,lam_d_KKT_new_no_u,v_y_d_new_no_u[:,:2],v_y_d_new_no_u[:,2:]):
    cart_vq = np.array(lambdified_vq_p_to_c(*tmpq,*tmpvq)).flatten()
    cart_vlam = np.array(lambdified_lamq_p_to_c(*tmpq,*tmpvq,*tmpvlam,*tmplam)).flatten()
    lam_d_KKT_new_no_u_cart.append(np.array(lambdified_lamv_p_to_c(*tmpq,*tmpvq,*tmpvlam,*tmplam)).flatten())
    v_y_d_new_no_u_cart.append(list(cart_vq) + list(cart_vlam))

v_y_d_new_no_u_cart = np.array(v_y_d_new_no_u_cart)
lam_d_KKT_new_no_u_cart = np.array(lam_d_KKT_new_no_u_cart)

def conserved_quantity_cartesian(q,lam,vq,vlam):
    x,y = np.array(q)
    lamx,lamy = np.array(lam)
    vx,vy = np.array(vq)
    vlamx,vlamy = np.array(vlam)

    return x*vlamy - y *vlamx -vx*lamy + vy*lamx  

I_new_evo = []
I_new_evo = p_y_d_new.T[1]
I_new_evo= np.array(I_new_evo,float)


I_new_no_u_evo = []
I_new_no_u_evo = p_y_d_new_no_u.T[1]
I_new_no_u_evo= np.array(I_new_no_u_evo,float)

    

In [ ]:
#standard computation of variables
kappa_standard = np.array([sympy.flatten(continuous_eq.kappa_calc(tmp)) for tmp in lamv_d_KKT],float)
vkappa_standard = []
for i in range(len(lamq_d_KKT)):
    vkappa_standard.append( sympy.flatten(continuous_eq.v_kappa_calc(q_d_KKT[i],vq_d_KKT[i],lamq_d_KKT[i],lamv_d_KKT[i],u_vec_KKT[i])))
vkappa_standard = np.array(vkappa_standard,float)

p_y_d_standard = np.array([sympy.flatten(tmp) for tmp in discrete_equations.p_v_d_from_y_d(q_d_KKT,kappa_standard,U1_d_KKT,U2_d_KKT,OCP_parameters_np_version)])


I_KKT_standard_evo = []
I_KKT_standard_evo = np.array(p_y_d_standard.T[1],float)

    

# plot results

In [ ]:
q_d_standard_KKT_sol_cart = []
q_d_new_KKT_sol_cart = []
q_d_new_KKT_sol_no_u_cart = []
import sys
from pathlib import Path 
foldername = 'a' + str(alpha_choice) + 'g' + str(gamma_choice)

for val in q_d_KKT:
    q_d_standard_KKT_sol_cart.append(polar_to_cartesian(val))
for val in q_d_KKT_new:
    q_d_new_KKT_sol_cart.append(polar_to_cartesian(val))  
for val in q_d_KKT_new_no_u:
    q_d_new_KKT_sol_no_u_cart.append(polar_to_cartesian(val))  
foldername += 'polar'

Path('figs/'+foldername).mkdir(parents=True, exist_ok=True)
    
q_d_standard_KKT_sol_cart = np.array(q_d_standard_KKT_sol_cart)
q_d_new_KKT_sol_cart = np.array(q_d_new_KKT_sol_cart)
q_d_new_KKT_sol_no_u_cart = np.array(q_d_new_KKT_sol_no_u_cart)



# Saving data

In [ ]:
if not standard_result_polar_KKT_new.success:
    print('did not find a solution in control dependent case, not storing the result')
else:
    storage_dict = dict()
    storage_dict["parameters"] = OCP_parameters_np_version
    storage_dict["q_d_new"] = q_d_KKT_new
    storage_dict["q_d_new_cartesian"] = q_d_new_KKT_sol_cart
    storage_dict["lambda_d_new"] = lam_d_KKT_new
    storage_dict["lambda_d_new_cart"] = lam_d_KKT_new_cart
    storage_dict["p_y_d_new"] = np.array(p_y_d_new,dtype=float)
    storage_dict["U1_d_new"] = np.array(U1_d_KKT_new,dtype=float)
    storage_dict["U2_d_new"] = np.array(U2_d_KKT_new,dtype=float)
    storage_dict["u_d_new"] = np.array(u_vec_KKT_new,dtype=float)
    storage_dict["I_d_new"] = np.array(p_y_d_new[:,1],dtype=float)
    storage_dict["H_control_new"] = np.array(H_control_KKT_new,dtype=float)
    storage_dict["v_y_d_new"] = np.array(v_y_d_new,dtype=float)
    storage_dict["v_y_d_new_cart"] = np.array(v_y_d_new_cart,dtype=float)
    storage_dict["q_d_standard_cart"] = q_d_standard_KKT_sol_cart
    storage_dict["q_d_standard"] =   q_d_KKT
    storage_dict["lambda_q_standard"] = lamq_d_KKT
    storage_dict["lambda_v_standard"] = lamv_d_KKT
    storage_dict["kappa_standard"] = kappa_standard
    storage_dict["vkappa_standard"] = vkappa_standard
    storage_dict["vq_d_standard"] = vq_d_KKT
    storage_dict["U1_d_standard"] = U1_d_KKT
    storage_dict["U2_d_standard"] = U2_d_KKT
    storage_dict["u_d_standard"] = u_vec_KKT
    storage_dict["I_d_standard"] = I_KKT_standard_evo
    storage_dict["H_pontry_standard"] = H_Pontry_KKT
    storage_dict['mu_new'] = mu_KKT_new
    storage_dict['nu_new'] = nu_KKT_new
    storage_dict['dt_new'] = dt_new
    storage_dict['Dt_new'] = Dt_new
    storage_dict['nfev_new'] = standard_result_polar_KKT_new.nfev
    storage_dict['dt_new_no_u'] = dt_new_no_u
    storage_dict['Dt_new_no_u'] = Dt_new_no_u
    storage_dict['nfev_new_no_u'] = standard_result_polar_KKT_new_no_u.nfev
    storage_dict['dt_standard'] = dt_standard
    storage_dict['Dt_standard'] = Dt_standard
    storage_dict['nfev_standard'] = standard_result_polar_KKT.nfev
    storage_dict["q_d_new_no_u"] = q_d_KKT_new_no_u
    storage_dict["q_d_new_no_u_cartesian"] = q_d_new_KKT_sol_no_u_cart
    storage_dict["lambda_d_new_no_u"] = lam_d_KKT_new_no_u
    storage_dict["lambda_d_new_no_u_cart"] = lam_d_KKT_new_no_u_cart
    storage_dict["v_y_d_new_no_u"] = np.array(v_y_d_new_no_u,dtype=float)
    storage_dict["v_y_d_new_no_u_cart"] = np.array(v_y_d_new_no_u_cart,dtype=float)
    storage_dict["p_y_d_new_no_u"] = np.array(p_y_d_new,dtype=float)
    storage_dict["u_d_new_no_u"] = np.array(u_d_new_no_u,dtype=float)
    storage_dict["I_d_new_no_u"] = np.array(I_new_no_u_evo,dtype=float)
    storage_dict["mu_new"] = mu_KKT_new
    storage_dict["nu_new"] = nu_KKT_new

    dirpath = "data/" + OCP_parameters_np_version["folder_name"]+"/data_"+"a=" + str(OCP_parameters_np_version["alpha"])  +"g=" + str(OCP_parameters_np_version["gamma"])
    Path(dirpath).mkdir(parents=True, exist_ok=True)

    file_name = dirpath+"/data_" +"a=" + str(OCP_parameters_np_version["alpha"])  +"g=" + str(OCP_parameters_np_version["gamma"])  +"N=" + str(OCP_parameters_np_version["N"]) + ".pkl"
    with open(file_name, 'wb') as ffile:
        pickle.dump(storage_dict, ffile)
        ffile.close()